# 03 - Modelling

High-value evaluation workflow: compare naive and regularized-linear baselines against LightGBM/CatBoost on 1-bar return forecasts using walk-forward validation, fit volatility (GARCH, Student-t) against realized-vol proxies with QLIKE, and export regime/anomaly/change-point/motif diagnostics. All preprocessing is fit on training data only.


In [1]:
## Setup

import os
os.environ.pop("SPARK_HOME", None)

import sys, warnings, yaml, joblib
from pathlib import Path
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler

_cwd = Path.cwd().resolve()
_candidates = list(dict.fromkeys([
    _cwd,
    *_cwd.parents,
    *(path.parent.parent for path in _cwd.glob("*/configs/pipeline.yaml")),
]))
_required_project_files = (
    "configs/pipeline.yaml",
    "src/features/pipeline.py",
    "notebooks/01_data_pipeline.ipynb",
    "Home.py",
)
_project_roots = [
    path for path in _candidates
    if all((path / relative).is_file() for relative in _required_project_files)
]
if not _project_roots:
    raise FileNotFoundError("Could not identify this project. Open its repository root or set the notebook working directory to it.")
if len(_project_roots) > 1:
    raise RuntimeError(f"Multiple project roots found: {_project_roots}. Open the intended repository root.")
PROJECT_ROOT = _project_roots[0]
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.baselines import naive_persistence, fit_arima, fit_garch, fit_linear_baseline
from src.models.evaluator import forecast_metrics, volatility_metrics
from src.models.lgbm_model import train_lgbm
from src.models.catboost_model import train_catboost
from src.models.regime_models import fit_gmm, fit_hmm, gmm_diagnostics, hmm_diagnostics
from src.models.anomaly_models import fit_isolation_forest, compute_matrix_profile, detect_changepoints_cusum
from src.models.rolling_validator import walk_forward_forecast
from src.visualisation.model_plots import plot_forecast_vs_actual, plot_feature_importance, plot_residuals
from src.visualisation.regime_plots import plot_regime_states, plot_volatility_regimes

with open(PROJECT_ROOT / 'configs' / 'pipeline.yaml', encoding='utf-8') as f:
    CFG = yaml.safe_load(f)

PROCESSED = PROJECT_ROOT / CFG['artifacts']['processed_dir']
MODELS    = PROJECT_ROOT / CFG['artifacts']['models_dir']
SCALERS   = PROJECT_ROOT / CFG['artifacts']['scalers_dir']
FIGURES   = PROJECT_ROOT / CFG['reports']['figures_dir']
METRICS   = PROJECT_ROOT / CFG['reports']['metrics_dir']
for d in [MODELS, SCALERS, FIGURES, METRICS]:
    d.mkdir(parents=True, exist_ok=True)

with open(PROCESSED / 'split_meta.yaml', encoding='utf-8') as f:
    SPLIT = yaml.safe_load(f)

FEATURE_COLS = SPLIT['feature_cols']
TARGET_COL   = SPLIT['target_col']
TS_COL       = SPLIT['ts_col']
print(f'Features : {len(FEATURE_COLS)}  |  Target : {TARGET_COL}')


Features : 62  |  Target : target_log_return_1


In [2]:
# Load partitions
train = pd.read_parquet(PROCESSED / 'train.parquet')
val = pd.read_parquet(PROCESSED / 'val.parquet')
test = pd.read_parquet(PROCESSED / 'test.parquet')
for partition in (train, val, test):
    partition[TS_COL] = pd.to_datetime(partition[TS_COL], utc=True)
    partition.sort_values(TS_COL, inplace=True)
    partition.reset_index(drop=True, inplace=True)

X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_val,   y_val   = val[FEATURE_COLS],   val[TARGET_COL]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET_COL]
print(f'Train {X_train.shape}  |  Val {X_val.shape}  |  Test {X_test.shape}')


Train (558601, 62)  |  Val (119999, 62)  |  Test (120000, 62)


In [3]:
# Shared result store and metrics (MAE/RMSE primary, sMAPE diagnostic only)
RESULTS = []  # list of dicts for flexible metrics/logging

def record_metrics(y_true, y_pred, model, partition, extra=None, y_train_ref=None, naive_pred=None):
    row = forecast_metrics(y_true, y_pred, label=f"{model}/{partition}", y_train=y_train_ref, naive_pred=naive_pred)
    row.update({"model": model, "partition": partition})
    if extra:
        row.update(extra)
    # replace existing same model+partition
    RESULTS[:] = [r for r in RESULTS if not (r.get('model')==model and r.get('partition')==partition)]
    RESULTS.append(row)
    return row

pd.DataFrame([{"note": "sMAPE is unstable near-zero for returns; prefer MAE/RMSE, direction accuracy, IC, MASE, naive-scaled MAE."}])


,note
0,sMAPE is unstable near-zero for returns; prefe...


---
## A - Baselines and walk-forward evaluation
Naive persistence and regularized linear models are the primary benchmarks. ARIMA is kept as a diagnostic benchmark only (1-bar returns often show little/no edge over naive).


In [4]:
# A1 - Naive / Persistence (benchmark)
history = pd.concat([train[TARGET_COL], val[TARGET_COL], test[TARGET_COL]], ignore_index=True)
naive_all = naive_persistence(history, horizon=1)
val_start, test_start = len(train), len(train) + len(val)
naive_val = naive_all.iloc[val_start:test_start]
naive_test = naive_all.iloc[test_start:]
record_metrics(y_val, naive_val, model='Naive', partition='Val', y_train_ref=y_train)
record_metrics(y_test, naive_test, model='Naive', partition='Test', y_train_ref=pd.concat([y_train, y_val]))

# Direction baseline (sign of last observed target as a naive directional view)
dir_col = None
for c in ['log_return', 'simple_return']:
    if c in test.columns:
        dir_col = c
        break


  Forecast [Naive/Val]  MAE=0.000795  RMSE=0.001728  sMAPE=147.57%  DA=0.489  IC=-0.006
  Forecast [Naive/Test]  MAE=0.001041  RMSE=0.002849  sMAPE=148.26%  DA=0.484  IC=0.006


In [5]:
# A2 - ARIMA(1,0,1) (diagnostic benchmark only)
try:
    arima_series = train['log_return'].dropna()
    print(f'Fitting ARIMA(1,0,1) on {len(arima_series):,} training obs...')
    arima_res = fit_arima(arima_series, order=(1,0,1), forecast_steps=len(y_val)+len(y_test))
    print(f'AIC={arima_res["aic"]:.2f}  BIC={arima_res["bic"]:.2f}')
    fc = np.asarray(arima_res['forecast']).reshape(-1)
    record_metrics(y_val,  fc[:len(y_val)], model='ARIMA(1,0,1)', partition='Val', naive_pred=naive_val.values, y_train_ref=y_train)
    record_metrics(y_test, fc[len(y_val):len(y_val)+len(y_test)], model='ARIMA(1,0,1)', partition='Test',
                   naive_pred=naive_test.values, y_train_ref=pd.concat([y_train, y_val]))
except Exception as e:
    print('ARIMA skipped:', e)


Fitting ARIMA(1,0,1) on 558,601 training obs...
AIC=-5362405.47  BIC=-5362360.53
  Forecast [ARIMA(1,0,1)/Val]  MAE=0.000538  RMSE=0.001218  sMAPE=196.74%  DA=0.491  IC=0.002
  Forecast [ARIMA(1,0,1)/Test]  MAE=0.000704  RMSE=0.002021  sMAPE=197.27%  DA=0.499  IC=0.000


In [6]:
# A3 - Regularized linear baseline (ridge)
try:
    lin = fit_linear_baseline(X_train, y_train, X_val, y_val, model_type='ridge', alpha=1.0)
    lin_val_pred = lin['val_pred']
    lin_test_pred = lin['model'].predict(X_test)
    record_metrics(y_val, lin_val_pred, model='Ridge', partition='Val', naive_pred=naive_val.values, y_train_ref=y_train)
    record_metrics(y_test, lin_test_pred, model='Ridge', partition='Test', naive_pred=naive_test.values, y_train_ref=pd.concat([y_train, y_val]))
    joblib.dump(lin, MODELS / 'ridge_baseline.pkl')
    print('Saved ridge baseline artifact.')
except Exception as e:
    print('Linear baseline skipped:', e)


  Forecast [Ridge/Val]  MAE=0.000539  RMSE=0.001222  sMAPE=172.84%  DA=0.513  IC=0.006
  Forecast [Ridge/Test]  MAE=0.000704  RMSE=0.002035  sMAPE=171.26%  DA=0.518  IC=-0.009
Saved ridge baseline artifact.


In [7]:
# A4 - Walk-forward evaluation (expanding window) vs naive
# Uses a rolling-origin split on train+val+test features to assess stability/fold spread.
try:
    wf_frame = pd.concat([train, val, test], ignore_index=True).dropna(subset=FEATURE_COLS + [TARGET_COL]).reset_index(drop=True)
    X_wf, y_wf = wf_frame[FEATURE_COLS], wf_frame[TARGET_COL]
    n = len(wf_frame)
    initial_train_size = max(int(n * 0.5), 200)
    test_size = max(int(n * 0.1), 50)

    def ridge_fit_predict(Xtr, ytr, Xte):
        m = fit_linear_baseline(Xtr, ytr, model_type='ridge', alpha=1.0)['model']
        return m.predict(Xte)

    def naive_fit_predict(Xtr, ytr, Xte):
        # persistence within evaluation fold context: last seen target for first point, else shifted actuals not known; use train tail
        last = float(np.asarray(ytr)[-1]) if len(ytr) else 0.0
        return np.full(len(Xte), last)

    wf_ridge = walk_forward_forecast(X_wf, y_wf, ridge_fit_predict, initial_train_size=initial_train_size, test_size=test_size, step_size=test_size, expanding=True)
    wf_naive = walk_forward_forecast(X_wf, y_wf, naive_fit_predict, initial_train_size=initial_train_size, test_size=test_size, step_size=test_size, expanding=True)
    wf_summary = {
        'initial_train_size': initial_train_size,
        'test_size': test_size,
        'step_size': test_size,
        'expanding': True,
        'ridge': wf_ridge,
        'naive_proxy': wf_naive,
    }
    (METRICS / 'walk_forward_summary.json').write_text(__import__('json').dumps(wf_summary, indent=2, default=float), encoding='utf-8')
    pd.DataFrame(wf_ridge.get('folds', [])).to_csv(METRICS / 'walk_forward_ridge_folds.csv', index=False)
    print('Walk-forward aggregate (ridge):', wf_ridge.get('aggregate'))
    print('Walk-forward aggregate (naive-proxy):', wf_naive.get('aggregate'))
except Exception as e:
    print('Walk-forward evaluation skipped:', e)


Walk-forward aggregate (ridge): {'mae': 0.0005986974390153659, 'rmse': 0.0015145541724262074, 'smape': 171.0431711121742, 'directional_accuracy': 0.511768173778967, 'correlation': -0.006997513176460801, 'mase': 0.6846873821110562, 'naive_scaled_mae': 0.6846880277295313, 'n_folds': 5}
Walk-forward aggregate (naive-proxy): {'mae': 0.0009440375458787058, 'rmse': 0.0017198711507380865, 'smape': 148.24630323068033, 'directional_accuracy': 0.5009472989383198, 'correlation': 0.0008003301115309157, 'mase': 1.0796281289682412, 'naive_scaled_mae': 1.0796293427489343, 'n_folds': 5}


---
## B - ML Models (tabular baselines)
LightGBM/CatBoost are evaluated as exploratory tabular models; compare against naive/linear and walk-forward results before claiming lift.


In [8]:
# B1 - LightGBM
lgbm_val_pred = lgbm_test_pred = None
try:
    print('Training LightGBM...')
    lgbm_res = train_lgbm(X_train, y_train, X_val, y_val, task='regression', artifact_name='lgbm_return')
    lgbm_model, lgbm_scaler = lgbm_res['model'], lgbm_res['scaler']
    lgbm_val_pred = lgbm_model.predict(lgbm_scaler.transform(X_val))
    lgbm_test_pred = lgbm_model.predict(lgbm_scaler.transform(X_test))
    record_metrics(y_val, lgbm_val_pred, model='LightGBM', partition='Val', naive_pred=naive_val.values, y_train_ref=y_train)
    record_metrics(y_test, lgbm_test_pred, model='LightGBM', partition='Test', naive_pred=naive_test.values, y_train_ref=pd.concat([y_train, y_val]))

    fig = plot_forecast_vs_actual(y_test, pd.Series(lgbm_test_pred, index=y_test.index),
                                  index=test.loc[y_test.index, TS_COL], title='LightGBM - Test Forecast vs Actual')
    fig.write_html(str(FIGURES / 'lgbm_forecast.html'), include_plotlyjs='cdn')
    fig_imp = plot_feature_importance(FEATURE_COLS, lgbm_model.feature_importances_, top_n=25, title='LightGBM - Feature Importance')
    fig_imp.write_html(str(FIGURES / 'lgbm_feature_importance.html'), include_plotlyjs='cdn')
    pd.DataFrame({'timestamp': test.loc[y_test.index, TS_COL].values, 'actual': y_test.values, 'predicted': lgbm_test_pred, 'naive': naive_test.loc[y_test.index].values}).to_csv(MODELS / 'lgbm_test_predictions.csv', index=False)
except Exception as e:
    print('LightGBM skipped:', e)


Training LightGBM...
  LightGBM artifact saved → artifacts\models\lgbm_return.pkl


c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  Forecast [LightGBM/Val]  MAE=0.000538  RMSE=0.001217  sMAPE=193.27%  DA=0.500  IC=0.045
  Forecast [LightGBM/Test]  MAE=0.000704  RMSE=0.002025  sMAPE=193.13%  DA=0.498  IC=0.006
LightGBM skipped: 'None of [RangeIndex(start=0, stop=120000, step=1)] are in the [index]'


In [9]:
# B2 - CatBoost
cb_val_pred = cb_test_pred = None
try:
    print('Training CatBoost...')
    cb_res = train_catboost(X_train, y_train, X_val, y_val, task='regression', artifact_name='catboost_return')
    cb_model, cb_scaler = cb_res['model'], cb_res['scaler']
    cb_val_pred = cb_model.predict(cb_scaler.transform(X_val))
    cb_test_pred = cb_model.predict(cb_scaler.transform(X_test))
    record_metrics(y_val, cb_val_pred, model='CatBoost', partition='Val', naive_pred=naive_val.values, y_train_ref=y_train)
    record_metrics(y_test, cb_test_pred, model='CatBoost', partition='Test', naive_pred=naive_test.values, y_train_ref=pd.concat([y_train, y_val]))

    fig = plot_forecast_vs_actual(y_test, pd.Series(cb_test_pred, index=y_test.index),
                                  index=test.loc[y_test.index, TS_COL], title='CatBoost - Test Forecast vs Actual')
    fig.write_html(str(FIGURES / 'catboost_forecast.html'), include_plotlyjs='cdn')
    pd.DataFrame({'timestamp': test.loc[y_test.index, TS_COL].values, 'actual': y_test.values, 'predicted': cb_test_pred, 'naive': naive_test.loc[y_test.index].values}).to_csv(MODELS / 'catboost_test_predictions.csv', index=False)
except Exception as e:
    print('CatBoost skipped:', e)


Training CatBoost...
  CatBoost artifact saved → artifacts\models\catboost_return.pkl
  Forecast [CatBoost/Val]  MAE=0.000538  RMSE=0.001218  sMAPE=191.11%  DA=0.504  IC=0.035
  Forecast [CatBoost/Test]  MAE=0.000704  RMSE=0.002021  sMAPE=191.68%  DA=0.496  IC=0.010
CatBoost skipped: 'None of [RangeIndex(start=0, stop=120000, step=1)] are in the [index]'


---
## C - Volatility (higher-signal risk target)
GARCH is fit with Student-t innovations (crypto fat tails) and compared to a realized-vol proxy using QLIKE/MAE.


In [10]:
# C1 - GARCH(1,1) with Student-t and realized-vol proxy evaluation
try:
    garch_cfg = CFG.get('models', {}).get('garch', {})
    garch_series = train['log_return'].dropna()
    print(f"Fitting GARCH(1,1) dist={garch_cfg.get('dist','t')} on {len(garch_series):,} training obs...")
    garch_res = fit_garch(garch_series, p=garch_cfg.get('p',1), q=garch_cfg.get('q',1), dist=garch_cfg.get('dist','t'), forecast_horizon=1)
    print(f'AIC={garch_res["aic"]:.2f}  BIC={garch_res["bic"]:.2f}')

    cond_vol = garch_res['conditional_vol']
    cv_index = train['log_return'].dropna().index
    cv_df = pd.DataFrame({TS_COL: train.loc[cv_index, TS_COL].values, 'cond_vol': cond_vol.values[:len(cv_index)]})
    cv_df.to_csv(MODELS / 'garch_cond_vol.csv', index=False)

    # Realized-vol proxy comparison (diagnostic; prefer exported realized target in production runs)
    rv_col = next((c for c in ['rolling_vol_30'] + [c for c in train.columns if c.startswith('rolling_vol_') and 'ann' not in c]), None)
    if rv_col and rv_col in train.columns:
        aligned = pd.merge_asof(cv_df.sort_values(TS_COL), train[[TS_COL, rv_col]].dropna().sort_values(TS_COL), on=TS_COL, direction='nearest')
        aligned = aligned.dropna(subset=['cond_vol', rv_col])
        if len(aligned) >= 30:
            vm = volatility_metrics(aligned[rv_col].values, aligned['cond_vol'].values, label='GARCH-vs-proxy')
            vm_row = {'model': f"GARCH(1,1)-{garch_res.get('distribution','t')}", 'partition': 'Train-proxy', **vm, 'realized_proxy': rv_col}
            RESULTS[:] = [r for r in RESULTS if not (r.get('model')==vm_row['model'] and r.get('partition')==vm_row['partition'])]
            RESULTS.append(vm_row)
            aligned.rename(columns={rv_col: 'realized_vol_proxy'}).to_csv(MODELS / 'garch_realized_comparison.csv', index=False)
    print('GARCH conditional vol saved.')
except Exception as e:
    print('GARCH skipped:', e)


Fitting GARCH(1,1) dist=t on 558,601 training obs...


c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\arch\univariate\base.py:694: DataScaleWarning:

y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.0399. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.


C:\Users\hp\Documents\GitHub\Crypto_TimeSeries_Analysis\src\models\baselines.py:73: ConvergenceWarning:

The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.




AIC=-1007434.60  BIC=-1007378.44
GARCH skipped: incompatible merge keys [0] dtype('<M8[ns]') and datetime64[ns, UTC], must be the same type


---
## D - Regime Discovery (exploratory, with diagnostics)
Unsupervised states are not ground truth; report BIC/log-likelihood, shares, persistence/switches, and inspect regime-conditioned behavior.


In [11]:
# D1 - Prepare regime features and scaler
regime_fcols = [c for c in FEATURE_COLS if any(x in c for x in ('rolling_vol','log_return','momentum','rolling_return'))]
if not regime_fcols:
    regime_fcols = [c for c in FEATURE_COLS if 'vol' in c or 'return' in c][:20]
print(f'Regime features ({len(regime_fcols)}): {regime_fcols[:10]}{"..." if len(regime_fcols)>10 else ""}')
df_full = pd.read_parquet(PROCESSED / 'btc_features.parquet').sort_values(TS_COL).reset_index(drop=True)
df_full[TS_COL] = pd.to_datetime(df_full[TS_COL], utc=True)
X_reg_train = train[regime_fcols].dropna().values
X_reg_full = df_full[regime_fcols].dropna().values
regime_scaler = StandardScaler()
X_reg_train_s = regime_scaler.fit_transform(X_reg_train)
X_reg_full_s = regime_scaler.transform(X_reg_full)
joblib.dump(regime_scaler, SCALERS / 'regime_scaler.pkl')
regime_index = df_full[regime_fcols].dropna().index
regime_diag = {}


Regime features (21): ['log_return', 'rolling_return_7', 'rolling_return_14', 'rolling_return_30', 'rolling_return_60', 'rolling_vol_7', 'rolling_vol_ann_7', 'rolling_vol_14', 'rolling_vol_ann_14', 'rolling_vol_30']...


In [12]:
# D2 - GMM
try:
    print('Fitting GMM...')
    gmm_res = fit_gmm(X_reg_train_s, artifact_name='gmm')
    gmm_states = gmm_res['model'].predict(X_reg_full_s)
    df_full.loc[regime_index, 'gmm_regime'] = gmm_states.astype(int)
    gd = gmm_diagnostics(X_reg_train_s, gmm_res['model'], gmm_res['labels'])
    regime_diag['gmm'] = gd
    print('GMM diagnostics:', gd)
    fig = plot_regime_states(df_full.loc[regime_index], ts_col=TS_COL, state_col='gmm_regime', title='GMM Market Regimes')
    fig.write_html(str(FIGURES / 'gmm_regimes.html'), include_plotlyjs='cdn')
except Exception as e:
    print('GMM skipped:', e)


Fitting GMM...
GMM skipped: Unable to allocate 89.5 MiB for an array with shape (21, 558601) and data type float64


In [13]:
# D3 - HMM
try:
    print('Fitting HMM...')
    hmm_res = fit_hmm(X_reg_train_s, artifact_name='hmm')
    hmm_states = hmm_res['model'].predict(X_reg_full_s)
    df_full.loc[regime_index, 'hmm_state'] = hmm_states.astype(int)
    hd = hmm_diagnostics(X_reg_train_s, hmm_res['model'], hmm_res['states'])
    regime_diag['hmm'] = hd
    print('HMM diagnostics:', hd)
    if hasattr(hmm_res['model'], 'transmat_'):
        n = hmm_res['model'].n_components
        print(pd.DataFrame(hmm_res['model'].transmat_, columns=[f'to_{i}' for i in range(n)], index=[f'from_{i}' for i in range(n)]).round(3))
    fig = plot_regime_states(df_full.loc[regime_index], ts_col=TS_COL, state_col='hmm_state', title='HMM Market States')
    fig.write_html(str(FIGURES / 'hmm_states.html'), include_plotlyjs='cdn')
except Exception as e:
    print('HMM skipped:', e)

(METRICS / 'regime_diagnostics.json').write_text(__import__('json').dumps(regime_diag, indent=2, default=float), encoding='utf-8')


Fitting HMM...
HMM skipped: Unable to allocate 12.8 MiB for an array with shape (3, 558601) and data type float64


2

---
## E - Anomaly, Motif/Discord, and Change-Point Detection
Exploratory signals: validate anomaly thresholds, motif/discord usefulness, and breakpoint sensitivity.


In [14]:
# E1 - Isolation Forest
try:
    print('Fitting Isolation Forest...')
    X_anom_train = train[FEATURE_COLS].dropna().values
    X_anom_full = df_full[FEATURE_COLS].dropna().values
    anom_scaler = StandardScaler()
    X_anom_tr_s = anom_scaler.fit_transform(X_anom_train)
    X_anom_full_s = anom_scaler.transform(X_anom_full)
    joblib.dump(anom_scaler, SCALERS / 'anomaly_scaler.pkl')
    iso_res = fit_isolation_forest(X_anom_tr_s, X_anom_full_s, artifact_name='isolation_forest')
    anom_index = df_full[FEATURE_COLS].dropna().index
    scores_df = pd.DataFrame({TS_COL: df_full.loc[anom_index, TS_COL].values, 'score': iso_res['full_scores'], 'is_anomaly': iso_res['anomaly_flags'].astype(int)})
    scores_df.to_csv(MODELS / 'anomaly_scores.csv', index=False)
    print(f'Anomalies flagged: {int(iso_res["anomaly_flags"].sum()):,} ({iso_res["anomaly_flags"].mean()*100:.2f}%)')
    fig = go.Figure([
        go.Scatter(x=pd.to_datetime(scores_df[TS_COL]), y=scores_df['score'], name='IF Score', line=dict(color='#4FC3F7', width=0.8)),
        go.Scatter(x=pd.to_datetime(scores_df[scores_df['is_anomaly']==1][TS_COL]), y=scores_df[scores_df['is_anomaly']==1]['score'],
                   mode='markers', name='Anomaly', marker=dict(color='#EF5350', size=5, symbol='x')),
    ])
    fig.update_layout(title='Isolation Forest - Anomaly Scores', template='plotly_dark')
    fig.write_html(str(FIGURES / 'anomaly_scores.html'), include_plotlyjs='cdn')
except Exception as e:
    print('Isolation Forest skipped:', e)


Fitting Isolation Forest...
Isolation Forest skipped: Unable to allocate 264. MiB for an array with shape (62, 558601) and data type float64


In [15]:
# E2 - Matrix Profile (motif/discord, exploratory)
try:
    MP_MAX_ROWS = 50_000
    lr_series_full = train['log_return'].dropna().iloc[-MP_MAX_ROWS:]
    mp_res = compute_matrix_profile(lr_series_full.values, window=30)
    disc_idx = int(mp_res['discord_idx'])
    mp_vals = np.asarray(mp_res['mp'])
    mp_ts = train.loc[lr_series_full.index[:len(mp_vals)], TS_COL].values
    mp_df = pd.DataFrame({TS_COL: pd.to_datetime(mp_ts), 'matrix_profile': mp_vals})
    mp_df.to_csv(MODELS / 'matrix_profile.csv', index=False)
    (MODELS / 'matrix_profile_meta.json').write_text(__import__('json').dumps({
        'window': 30, 'discord_idx': disc_idx,
        'discord_timestamp': str(train.loc[lr_series_full.index[min(disc_idx, len(lr_series_full.index)-1)], TS_COL]),
        'discord_value': float(mp_res['discord_value']),
        'rows_used': int(len(lr_series_full)),
    }, indent=2), encoding='utf-8')
    fig = go.Figure([go.Scatter(x=pd.to_datetime(mp_ts), y=mp_vals, name='MP', line=dict(color='#F7931A'))])
    if disc_idx < len(mp_ts):
        fig.add_vline(x=pd.to_datetime(mp_ts[disc_idx]), line_color='#EF5350', line_dash='dot')
    fig.update_layout(title='Matrix Profile - Log Returns (window=30)', template='plotly_dark')
    fig.write_html(str(FIGURES / 'matrix_profile.html'), include_plotlyjs='cdn')
    print('Matrix profile exported.')
except Exception as e:
    print('Matrix Profile skipped:', e)


Matrix profile exported.


In [16]:
# E3 - CUSUM Change-Point Detection (calibrated on training partition)
try:
    cusum_series = train['log_return'].dropna().values
    print(f'Detecting change-points on {len(cusum_series):,} training observations...')
    bkps = detect_changepoints_cusum(cusum_series, n_bkps=5, model='rbf', max_samples=3_000)
    train_ts_index = train['log_return'].dropna().index
    bp_records = []
    for bp in bkps[:-1]:
        if bp < len(train_ts_index):
            ts_val = train.loc[train_ts_index[bp], TS_COL]
            bp_records.append({'index': int(bp), 'timestamp': str(ts_val)})
            print(f'  Breakpoint {bp} -> {ts_val}')
    pd.DataFrame(bp_records).to_csv(MODELS / 'changepoints.csv', index=False)
    vol_col = next((c for c in FEATURE_COLS if 'rolling_vol_' in c), 'log_return')
    fig = plot_volatility_regimes(train, vol_col=vol_col, ts_col=TS_COL,
                                  changepoints=[r['index'] for r in bp_records] + [len(train)],
                                  title='CUSUM Change-Points on Rolling Volatility')
    fig.write_html(str(FIGURES / 'cusum_changepoints.html'), include_plotlyjs='cdn')
except Exception as e:
    print('CUSUM skipped:', e)


Detecting change-points on 558,601 training observations...
  Breakpoint 66385 -> 2017-11-10 10:37:00+00:00
  Breakpoint 216920 -> 2018-05-18 12:23:00+00:00
  Breakpoint 502095 -> 2019-05-10 02:16:00+00:00


In [17]:
# Save metrics and summary for notebook 04 / dashboard
metrics_df = pd.DataFrame(RESULTS)
metrics_path = PROCESSED / 'metrics_draft.yaml'
with open(metrics_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(RESULTS, f, sort_keys=False)
metrics_df.to_csv(METRICS / 'forecast_metrics.csv', index=False)
print(f'Interim metrics saved -> {metrics_path}')
print(metrics_df[[c for c in ['model','partition','mae','rmse','directional_accuracy','correlation','mase','naive_scaled_mae','qlike'] if c in metrics_df.columns]].head(20).to_string(index=False))
print('Next -> 04_evaluation_and_export.ipynb')


Interim metrics saved -> C:\Users\hp\Documents\GitHub\Crypto_TimeSeries_Analysis\processed\metrics_draft.yaml
       model partition      mae     rmse  directional_accuracy   correlation     mase  naive_scaled_mae
       Naive       Val 0.000795 0.001728              0.489207 -6.181868e-03 0.589895          1.000000
       Naive      Test 0.001041 0.002849              0.483832  6.423294e-03 0.833538          1.000000
ARIMA(1,0,1)       Val 0.000538 0.001218              0.491262  2.463848e-03 0.399435          0.677153
ARIMA(1,0,1)      Test 0.000704 0.002021              0.498897  3.447933e-18 0.563264          0.675750
       Ridge       Val 0.000539 0.001222              0.513088  5.515478e-03 0.400015          0.678128
       Ridge      Test 0.000704 0.002035              0.518432 -9.458082e-03 0.563866          0.676471
    LightGBM       Val 0.000538 0.001217              0.500264  4.499246e-02 0.399349          0.677007
    LightGBM      Test 0.000704 0.002025              0.49